### Регрессия с полным историческим хвостом fact_value

Модель предсказывает fact_value на полгода 2025 (июль–декабрь), используя **все предыдущие значения fact_value** за всю историю по каждой операции.

* подготовка данных
  * добавление lag-признаков: lag_1, lag_2, ..., lag_N — весь исторический хвост fact_value
  * проверка на пропуски данных
  * построение графиков
  * разделение по времени: тест = июль–декабрь 2025
* обучение модели
  * обучить **одну** модель **LinearRegression** на **все операции** resist_id
  * `operation_id` + все lag-признаки (исторический хвост) используются для предсказания
* оценка точности модели
  * рассчитать Точность, Стабильность, Качество по каждой операции и сводно
* вывести результаты в excel
* выводы по работе

In [ ]:
# импорт библиотек
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")

#### Конфигурация

Выбираем **один** resist_id. Модель обучается на **все** его операции (operation_id используется как числовой признак).

In [ ]:
# --- Выбор resist_id (можно менять) ---
RESIST_ID = 124

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]  # июль — декабрь 2025

#### Загрузка данных, оценка и подготовка

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_division_title(div_json: str):
    if pd.isna(div_json):
        return None
    try:
        obj = json.loads(div_json)
    except Exception:
        return None
    return obj.get("name")


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def get_calc_value(calc_obj: dict, key: str):
    val = calc_obj.get(key)
    if isinstance(val, dict):
        return val.get("value")
    return val


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    """Разворачивает calculate_parameter в плоский словарь: все поля → колонки calc_<key>."""
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


# Загрузка CSV
dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(
        f"Файлы данных не найдены в {RAW_DIR}. "
        "Убедитесь, что папка data/raw существует и содержит data-2022.csv и др."
    )

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            # "division_title": parse_division_title(row.get("division_name")),
            # "id": row.get("id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record})  # , **calc_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

# Приведение calc_* колонок к числу (где возможно)
calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = pd.to_numeric(df_model39[col], errors="coerce")
print("Развёрнутые поля calculate_parameter:", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

# Фильтр: только выбранный resist_id (все его операции)
df = df_model39[df_model39["resist_id"] == RESIST_ID].copy()

if df.empty:
    raise ValueError(
        f"Нет данных для resist_id={RESIST_ID}. "
        f"Доступные resist_id: {sorted(df_model39['resist_id'].dropna().unique()[:10].tolist())}..."
    )

# operation_id → числовой код для LinearRegression
df["op_code"] = df["operation_id"].astype("category").cat.codes

print(f"resist_id={RESIST_ID}: {df['operation_id'].nunique()} операций, {len(df)} строк")

# Feature Engineering: ВСЕ предыдущие fact_value за всю историю (lag-признаки)
# lag_1 = значение 1 месяц назад, lag_2 = 2 месяца назад, ... — весь исторический хвост
group = df.groupby("operation_id")
max_lags = max(group.size()) - 1  # макс. лагов = длина ряда минус 1 (для первой строки нет истории)
max_lags = max(1, max_lags)
LAG_COLS = [f"lag_{i}" for i in range(1, max_lags + 1)]
for i in range(1, max_lags + 1):
    df[f"lag_{i}"] = group["fact_value"].shift(i)
print(f"Добавлено {len(LAG_COLS)} lag-признаков (весь исторический хвост fact_value)")

num_cols_clean = df.select_dtypes(include=["number"]).columns
df[num_cols_clean] = df[num_cols_clean].fillna(0)
df = df.infer_objects(copy=False)
df.columns
# df.head()

#### Проверка пропусков данных

In [ ]:
# проверка пропусков значений
if df.isnull().sum().sum() != 0:
    print("Обнаружены пропуски:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    df = df.fillna(0)
else:
    print("dataset не содержит пустых значений")

#### Оценка датасета и построение графиков

In [ ]:
# df.describe().T

In [ ]:
# График: fact_value (количество операций) по времени — по каждой операции
df_plot = df.copy()
df_plot["date_plot"] = pd.to_datetime(df_plot["date"])
plt.figure(figsize=(12, 5))
for op_id, grp in df_plot.groupby("operation_id"):
    grp_sorted = grp.sort_values("date_plot")
    plt.plot(grp_sorted["date_plot"], grp_sorted["fact_value"], marker="o", markersize=3, label=f"op {op_id}")
plt.xlabel("Дата")
plt.ylabel("fact_value (количество операций)")
plt.title(f"Динамика fact_value для resist_id={RESIST_ID} (все операции)")
plt.xticks(rotation=45)
plt.legend(fontsize=7, ncol=2, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Распределение целевой переменной
sns.histplot(df["fact_value"], bins=15, kde=True)
plt.xlabel("fact_value")
plt.ylabel("Частота")
plt.title("Распределение количества операций")
plt.show()

#### Разделение по времени: тест = июль–декабрь 2025 (все операции resist_id)

In [ ]:
FEATURE_COLS = ["op_code", "month", "year", "fact_predict",
                "workDayMonth_2", "calculate_minute"] + LAG_COLS

X = df[FEATURE_COLS]
y = df["fact_value"]

# Тест: год=2025 и месяц в [7..12]
mask_test = (df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS))
mask_train = ~mask_test

# Отбрасываем строки без достаточной истории (хотя бы 1 lag)
mask_has_history = df[LAG_COLS].notna().any(axis=1)
mask_train = mask_train & mask_has_history

if mask_train.sum() == 0:
    raise ValueError(f"Нет обучающих данных (до {TEST_YEAR}-{min(TEST_MONTHS):02d})")
if mask_test.sum() == 0:
    raise ValueError(f"Нет тестовых данных за {TEST_YEAR}, месяцы {TEST_MONTHS}")

X_train = X.loc[mask_train]
X_test = X.loc[mask_test]
y_train = y.loc[mask_train]
y_test = y.loc[mask_test]

print(f"Обучающая выборка: {len(X_train)} строк ({df.loc[mask_train, 'operation_id'].nunique()} операций)")
print(f"Тестовая выборка:  {len(X_test)} строк ({df.loc[mask_test, 'operation_id'].nunique()} операций)")

#### Обучение модели LinearRegression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

#### Оценка точности предсказания

In [ ]:
y_pred = model.predict(X_test)
y_pred = np.maximum(y_pred, 0)

df_test = df.loc[X_test.index].copy()
df_test["predict_linear"] = y_pred

y_true = np.asarray(y_test, dtype=float)
y_pred_arr = np.asarray(y_pred, dtype=float)

# Сводные метрики по всем операциям resist_id
acc = np.empty_like(y_true)
mask_zero = y_true == 0
acc[mask_zero] = np.where(y_pred_arr[mask_zero] == 0, 100.0, 0.0)
mask_ok = ~mask_zero
if np.any(mask_ok):
    acc[mask_ok] = np.clip(
        100.0 - np.abs(y_true[mask_ok] - y_pred_arr[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
        0, 100,
    )

tochnost = float(np.mean(acc))
stab = 0.0
if tochnost > 0:
    cv = np.std(acc, ddof=0) / tochnost
    stab = max(0.0, (1 - cv)) * 100
kachestvo = 0.0
if tochnost + stab > 0:
    kachestvo = 2 * tochnost * stab / (tochnost + stab)

print(f"=== Сводные метрики по resist_id={RESIST_ID} (все операции) ===")
print(f"Точность, %:     {tochnost:.2f}")
print(f"Стабильность, %: {stab:.2f}")
print(f"Качество, %:     {kachestvo:.2f}")
print(f"Строк в тесте:   {len(y_true)}, операций: {df_test['operation_id'].nunique()}")

#### Выводы по работе

* Одна модель **LinearRegression** обучена на **все операции** выбранного `RESIST_ID`. Признак `operation_id` закодирован в числовой `op_code`.
* **Точность** — среднее по «Точность прогноза» (100% − MAPE) по строкам. Чем выше, тем меньше относительная ошибка.
* **Стабильность** — (1 − CV) × 100%, где CV = σ/μ по «Точность прогноза». Чем меньше разброс точности, тем стабильнее.
* **Качество** — гармоническое среднее Точности и Стабильности. Падение любого показателя снижает рейтинг.

Сравнение можно проводить, меняя `RESIST_ID` или добавляя другие признаки.

In [ ]:
def calc_metrics_for_forecast(df_sub: pd.DataFrame, forecast_col: str) -> tuple[float, float, float]:
    """Точность, Стабильность, Качество по df с fact_value и forecast_col."""
    if df_sub.empty:
        return 0.0, 0.0, 0.0
    y_true = df_sub["fact_value"].astype(float).values
    y_pred = df_sub[forecast_col].astype(float).fillna(0).values
    y_pred = np.maximum(y_pred, 0)
    acc = np.empty_like(y_true, dtype=float)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0, 100,
        )
    tochnost = float(np.mean(acc))
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = max(0.0, (1 - cv)) * 100
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = 2 * tochnost * stab / (tochnost + stab)
    return round(tochnost, 2), round(stab, 2), round(kachestvo, 2)


# Таблица по операциям
rows_op = []
for (rid, op_id), grp in df_test.groupby(["resist_id", "operation_id"]):
    t1, s1, q1 = calc_metrics_for_forecast(grp, "fact_predict")
    t2, s2, q2 = calc_metrics_for_forecast(grp, "predict_linear")
    rows_op.append({
        "resist_id": rid,
        "operation_id": op_id,
        "Точность (fact_predict)": t1,
        "Стабильность (fact_predict)": s1,
        "Качество (fact_predict)": q1,
        "Точность (прогноз)": t2,
        "Стабильность (прогноз)": s2,
        "Качество (прогноз)": q2,
    })

# Сводная строка по resist_id (все операции вместе)
t1_all, s1_all, q1_all = calc_metrics_for_forecast(df_test, "fact_predict")
t2_all, s2_all, q2_all = calc_metrics_for_forecast(df_test, "predict_linear")
rows_rid = [{
    "resist_id": RESIST_ID,
    "Точность (fact_predict)": t1_all,
    "Стабильность (fact_predict)": s1_all,
    "Качество (fact_predict)": q1_all,
    "Точность (прогноз)": t2_all,
    "Стабильность (прогноз)": s2_all,
    "Качество (прогноз)": q2_all,
}]

df_metrics_op = pd.DataFrame(rows_op)
df_metrics_rid = pd.DataFrame(rows_rid)

# Детали тестовой выборки
df_detail = df_test[["resist_id", "operation_id", "year", "month", "fact_value", "fact_predict", "predict_linear"]].copy()
df_detail = df_detail.sort_values(["resist_id", "operation_id", "year", "month"]).reset_index(drop=True)
df_detail = df_detail.rename(columns={
    "fact_value": "fact_value (факт)",
    "fact_predict": "fact_predict (кол-во)",
    "predict_linear": "прогноз (кол-во)",
})

out_path = Path("../data/processed/regression_full_history_h2_2025.xlsx")
out_path.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    df_detail.to_excel(w, sheet_name="Данные теста", index=False)
    df_metrics_op.to_excel(w, sheet_name="По операциям", index=False)
    df_metrics_rid.to_excel(w, sheet_name="По resist_id", index=False)

print(f"Excel сохранён: {out_path.resolve()}")